In [0]:
# ============================================
# RAILWAY DATA ENGINEERING PROJECT
# DATA QUALITY & AUDIT
# ============================================

from pyspark.sql.functions import *

# Load Bronze
df_bronze = spark.table(
    "railway_data_engineering.bronze.railway_bronze"
)

# Load Silver
df_silver = spark.table(
    "railway_data_engineering.silver.railway_cleaned"
)

# Load Gold tables
df_station = spark.table(
    "railway_data_engineering.gold.station_summary"
)

df_route = spark.table(
    "railway_data_engineering.gold.route_summary"
)

df_day = spark.table(
    "railway_data_engineering.gold.day_summary"
)

df_day_type = spark.table(
    "railway_data_engineering.gold.day_type_summary"
)

print("✅ All Bronze, Silver and Gold tables loaded successfully!")

In [0]:
# ============================================
# RECORD COUNT AUDIT
# ============================================

bronze_count = df_bronze.count()
silver_count = df_silver.count()
gold_station_count = df_station.count()
gold_route_count = df_route.count()
gold_day_count = df_day.count()
gold_day_type_count = df_day_type.count()

print("Bronze records       :", bronze_count)
print("Silver records       :", silver_count)
print("Gold station records :", gold_station_count)
print("Gold route records   :", gold_route_count)
print("Gold day records     :", gold_day_count)
print("Gold day-type records:", gold_day_type_count)

In [0]:
# ============================================
# BRONZE → SILVER VALIDATION
# ============================================

if bronze_count == silver_count:
    print("✅ Bronze → Silver record count: PASSED")
else:
    print("❌ Bronze → Silver record count: FAILED")
    print("Difference:", bronze_count - silver_count)

In [0]:
# ============================================
# SILVER NULL VALIDATION
# ============================================


null_counts = df_silver.select([
    sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df_silver.columns
])

display(null_counts)

In [0]:
# ============================================
# BUSINESS RULE VALIDATION
# ============================================

valid_days = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

invalid_day_count = df_silver.filter(
    ~col("days_clean").isin(valid_days)
).count()

invalid_day_number_count = df_silver.filter(
    (col("day_number") < 1) |
    (col("day_number") > 7) |
    col("day_number").isNull()
).count()

invalid_day_type_count = df_silver.filter(
    ~col("day_type").isin("Weekday", "Weekend")
).count()

print("Invalid day records       :", invalid_day_count)
print("Invalid day_number records:", invalid_day_number_count)
print("Invalid day_type records :", invalid_day_type_count)

In [0]:
# ============================================
# AUDIT RESULT
# ============================================

pipeline_status = "PASSED"

if bronze_count != silver_count:
    pipeline_status = "FAILED"

if invalid_day_count != 0:
    pipeline_status = "FAILED"

if invalid_day_number_count != 0:
    pipeline_status = "FAILED"

if invalid_day_type_count != 0:
    pipeline_status = "FAILED"

audit_data = [
    (
        "Railway_ETL",
        bronze_count,
        silver_count,
        gold_station_count,
        gold_route_count,
        gold_day_count,
        gold_day_type_count,
        pipeline_status
    )
]

audit_df = spark.createDataFrame(
    audit_data,
    [
        "pipeline_name",
        "bronze_records",
        "silver_records",
        "gold_station_records",
        "gold_route_records",
        "gold_day_records",
        "gold_day_type_records",
        "pipeline_status"
    ]
)

audit_df = audit_df.withColumn(
    "audit_timestamp",
    current_timestamp()
)

display(audit_df)

In [0]:
# ============================================
# SAVE AUDIT TABLE
# ============================================

audit_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        "railway_data_engineering.gold.pipeline_audit"
    )

print("✅ Pipeline audit table created successfully!")

In [0]:
audit_check = spark.table(
    "railway_data_engineering.gold.pipeline_audit"
)

display(audit_check)